In [18]:
import pandas as pd
from pathlib import Path

In [19]:
def retriever_writer():
    # === CONFIG ===
    database_file = "/home/omar/git/logs/master_df.csv"

    # Multiple sources at once
    sources = ["R240121", "R240114"]
    telescope = "wb"

    # Where to save manifests
    manifest_dir = Path("manifests")
    manifest_dir.mkdir(exist_ok=True)

    # === Load DB and filter ===
    df = pd.read_csv(database_file)

    # Filter by source, telescope, and MJD_start_time <= 60919
    subset = df[
        (df["Source_Name"].isin(sources)) &
        (df["Telescope"] == telescope) &
        (df["MJD_start_time"] <= 60919)
    ].copy()

    if subset.empty:
        print("No observations match the criteria (sources, telescope, MJD filter). Exiting.")
        return

    # Replace .txt with .fil to match the archive
    subset["archive_filename"] = subset["complete_filename"].str.replace(".txt", ".fil", regex=False)

    # Build relative archive paths
    subset["rel_path"] = subset["scan_name"] + "/" + subset["archive_filename"]

    # === Extract only the date before 'T' and convert to YYYY_MM_DD ===
    subset["Obs_Date_only"] = subset["Obs_Date_String"].str.split("T").str[0]
    subset["date_str"] = pd.to_datetime(subset["Obs_Date_only"]).dt.strftime("%Y_%m_%d")

    # === Determine BAND based on Central_freq_(MHz) ===
    subset["BAND"] = subset["Central_freq_(MHz)"].apply(lambda f: "P" if f < 400 else "L")

    # === Group by experiment (scan_name) and source ===
    log_entries = []
    for (exp, src), group in subset.groupby(["scan_name", "Source_Name"]):
        # Take first observation date of this group
        date_str = group["date_str"].iloc[0]

        # Take first BAND (assuming all observations in the group have same band)
        band = group["BAND"].iloc[0]

        # Build manifest filename
        manifest_file = manifest_dir / f"{date_str}_{band}_manifest_{exp}_{src}.txt"

        # Write relative paths
        group["rel_path"].to_csv(manifest_file, index=False, header=False)

        log_entries.append(f"{exp} ({src}, {date_str}, {band}): {len(group)} files -> {manifest_file}")

    # === Write overview log ===
    overview_file = manifest_dir / "manifest_overview.log"
    with open(overview_file, "w") as f:
        f.write("\n".join(log_entries))

    print("Created per-experiment, per-source manifests with dates and band:")
    print("\n".join(log_entries))
    print(f"\nOverview written to {overview_file}")

    return

In [21]:
retriever_writer()

/tmp/ipykernel_2376486/50038818.py:14: DtypeWarning: Columns (11,34,38,39,40,43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(database_file)


Created per-experiment, per-source manifests with dates and band:
p47001 (R240114, 2024_02_09, P): 38 files -> manifests/2024_02_09_P_manifest_p47001_R240114.txt
p47002 (R240114, 2024_02_10, P): 38 files -> manifests/2024_02_10_P_manifest_p47002_R240114.txt
p47003 (R240114, 2024_02_11, P): 38 files -> manifests/2024_02_11_P_manifest_p47003_R240114.txt
p47004 (R240114, 2024_02_12, P): 38 files -> manifests/2024_02_12_P_manifest_p47004_R240114.txt
p47005 (R240114, 2024_02_13, P): 38 files -> manifests/2024_02_13_P_manifest_p47005_R240114.txt
p47006 (R240114, 2024_02_14, P): 38 files -> manifests/2024_02_14_P_manifest_p47006_R240114.txt
p47007 (R240114, 2024_02_16, P): 38 files -> manifests/2024_02_16_P_manifest_p47007_R240114.txt
p47008 (R240114, 2024_02_17, P): 38 files -> manifests/2024_02_17_P_manifest_p47008_R240114.txt
p47009 (R240114, 2024_02_18, P): 38 files -> manifests/2024_02_18_P_manifest_p47009_R240114.txt
p47010 (R240114, 2024_02_19, P): 38 files -> manifests/2024_02_19_P_ma